<a href="https://colab.research.google.com/github/plavebo/SafeAI/blob/main/face_filter_vF(YE).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 얼굴 데이터셋 정제

## 흐름
```
Drive 공유링크 zip
  → gdown으로 다운로드
  → 압축 해제
  → 자동 정제 (얼굴 감지 / 각도 검증 / 블러 / 해상도 / 중복 등)
  → clean/ 폴더에 저장
```

## Step 1. 패키지 설치하기

In [1]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-q_p805st
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-q_p805st
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=e04db09a4964b3727ed3ed88d99b13f8111c94c3b40cbba4d9c6e3c96137a6a5
  Stored in directory: /tmp/pip-ephem-wheel-cache-6n6p901c/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [2]:
!pip install -q gdown insightface onnxruntime-gpu imagehash glasses-detector

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('✅ 설치 완료')
print(f'GPU: {gpu if gpu else "❌ 없음 → 런타임 유형 변경에서 T4 GPU 선택하세요"}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 6.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 75.4 MB/s eta 0:00:00
✅ 설치 완료
GPU: Tesla T4


## Step 2. 설정

In [3]:
# ================================================================
# ① Google Drive 공유링크 붙여넣기
#    Drive에서 파일 우클릭 → 공유 → 링크 복사
# ================================================================
DRIVE_SHARE_URL = 'https://drive.google.com/file/d/1G9u2BrpyyiFsyUkMg2kfwWX7bvcNPMzY/view?usp=drive_link'

# ================================================================
# ② 경로 설정 (건드릴 필요 없음)
# ================================================================
ZIP_PATH  = '/content/dataset.zip'   # 다운받을 zip 위치
RAW_DIR   = '/content/raw'           # 압축 해제 위치
CLEAN_DIR = '/content/clean'         # 정제 결과 위치
REPORT    = '/content/report.csv'    # 결과 리포트

# ================================================================
# ③ 정제 파라미터
# ================================================================
MIN_FACE      = 80    # 얼굴 bbox 최소 px (80×80)
MIN_IMG       = 200   # 전체 이미지 최소 px (200×200)
BLUR_THR      = 80.0  # 블러 임계값 (낮출수록 관대)
YAW_THR       = 35.0  # 정면 판정 yaw 각도 (±35° 이내 = 정면)
SIM_THR       = 0.68  # 행사 중복 판정 유사도
MAX_PER_ANGLE = 3     # 같은 각도 최대 허용 장수

print('✅ 설정 완료')

✅ 설정 완료


## Step 3. Drive에서 zip 다운로드 + 압축 해제

In [4]:
import gdown, zipfile, os
from pathlib import Path

# 공유링크 → file ID 추출
if 'drive.google.com' in DRIVE_SHARE_URL:
    file_id = DRIVE_SHARE_URL.split('/d/')[1].split('/')[0]
    download_url = f'https://drive.google.com/uc?id={file_id}'
else:
    download_url = DRIVE_SHARE_URL

print(f'다운로드 중... (파일 크기에 따라 수분 소요)')
gdown.download(download_url, ZIP_PATH, quiet=False)
print(f'✅ 다운로드 완료: {ZIP_PATH}')

# 압축 해제
print('압축 해제 중...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(RAW_DIR)
print(f'✅ 압축 해제 완료: {RAW_DIR}')

# 구조 확인
raw_root = Path(RAW_DIR)
all_imgs = list(raw_root.rglob('*.jpg')) + list(raw_root.rglob('*.jpeg')) \
         + list(raw_root.rglob('*.png')) + list(raw_root.rglob('*.webp'))
print(f'\n총 이미지 수: {len(all_imgs)}장')
print('폴더 구조 샘플:')
for p in sorted(all_imgs)[:5]:
    print(f'  {p.relative_to(raw_root)}')

다운로드 중... (파일 크기에 따라 수분 소요)


Downloading...
From (original): https://drive.google.com/uc?id=1G9u2BrpyyiFsyUkMg2kfwWX7bvcNPMzY
From (redirected): https://drive.google.com/uc?id=1G9u2BrpyyiFsyUkMg2kfwWX7bvcNPMzY&confirm=t&uuid=cfe0d942-ae63-4787-815a-cc4d146d456d
To: /content/dataset.zip
100%|██████████| 136M/136M [00:02<00:00, 65.0MB/s]


✅ 다운로드 완료: /content/dataset.zip
압축 해제 중...
✅ 압축 해제 완료: /content/raw

총 이미지 수: 3000장
폴더 구조 샘플:
  데이터셋/한국 연예인/김고은/김고은_정면_1.jpg
  데이터셋/한국 연예인/김고은/김고은_정면_10.jpg
  데이터셋/한국 연예인/김고은/김고은_정면_100.jpg
  데이터셋/한국 연예인/김고은/김고은_정면_11.jpg
  데이터셋/한국 연예인/김고은/김고은_정면_12.jpg


## Step 4. 폴더 구조 자동 감지
zip 안 구조에 따라 자동으로 인물별 그룹핑합니다

- **폴더 있는 경우** → 폴더명 = 인물명
- **폴더 없이 파일만 있는 경우** → 파일명에서 인물명 추출 (`김고은_측면_1.jpg` → `김고은`)

In [5]:
from collections import defaultdict
from pathlib import Path

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
raw_root = Path(RAW_DIR)
person_images = defaultdict(list)

# 모든 이미지 파일 찾아서 부모 폴더명(=인물명)으로 그룹핑
for p in raw_root.rglob('*'):
    if p.suffix.lower() not in IMG_EXTS:
        continue
    person = p.parent.name  # 현아, 한소희, 카리나 ...
    person_images[person].append(p)

print(f'감지된 인물: {len(person_images)}명')
for person, imgs in sorted(person_images.items()):
    print(f'  {person}: {len(imgs)}장')

감지된 인물: 20명
  김고은: 150장
  나재민: 150장
  남궁민: 150장
  류진: 150장
  마동석: 150장
  박보검: 150장
  박지훈: 150장
  뷔: 150장
  비투비 이민혁: 150장
  설윤: 150장
  송강호: 150장
  수지: 150장
  유해진: 150장
  이영애: 150장
  이효리: 150장
  장원영: 150장
  카리나: 150장
  투바투 수빈: 150장
  한소희: 150장
  현아: 150장


## Step 5. 자동 정제 실행

각 이미지마다 아래를 자동으로 체크합니다:

| 체크 | 기준 | 탈락 시 |
|---|---|---|
| 전체 해상도 | 200×200 미만 | rejected/이미지해상도부족 |
| 얼굴 감지 | 얼굴 없음 | rejected/얼굴없음 |
| 다중 얼굴 | 2개 이상 | rejected/다중얼굴 |
| 얼굴 해상도 | 80×80 미만 | rejected/얼굴해상도부족 |
| 정면/측면 라벨 검증 | 파일명 vs 실제 yaw 각도 | rejected/각도불일치 |
| 블러/과보정 | Laplacian variance < 80 | rejected/블러 |
| 중복 사진 | pHash 해밍거리 < 8 | rejected/pHash중복 |
| 행사 중복 | 같은 각도 유사도 ≥ 0.68 초과 3장 | rejected/행사중복 |
| 안경 착용 | glasses-detector | attack/ 분리 |
| 마스크 착용 | CLIP zero-shot | rejected/마스크착용 |

In [6]:
# 모델 로드
import cv2, numpy as np, shutil, csv
from dataclasses import dataclass, field
from typing import Optional
from PIL import Image as PILImage
from imagehash import phash
from insightface.app import FaceAnalysis

#모델 초기화
face_app = FaceAnalysis(name='buffalo_l', #buffalo_1은 insightface의 고정밀 모델이다
                        providers=['CUDAExecutionProvider','CPUExecutionProvider'])
face_app.prepare(ctx_id=0, det_size=(640,640)) #det_size는 640*640 해상도를 의미함
print('✅ insightface 로드')

try:
    from glasses_detector import GlassesClassifier
    glasses_clf = GlassesClassifier()
    GLASSES_OK = True
    print('✅ glasses-detector 로드')
except Exception as e:
    GLASSES_OK = False
    print(f'⚠️  glasses-detector 실패: {e}')

try:
    import clip, torch
    clip_model, clip_prep = clip.load('ViT-B/32', device='cpu')
    clip_mask_tok = clip.tokenize(['face without mask', 'face with mask or covering'])
    CLIP_OK = True
    print('✅ CLIP 로드')
except Exception as e:
    CLIP_OK = False
    print(f'⚠️  CLIP 실패 (마스크 체크 스킵): {e}')

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:06<00:00, 42363.71KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

100%|██████████| 4.96M/4.96M [00:00<00:00, 89.7MB/s]


✅ glasses-detector 로드


100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 107MiB/s]


✅ CLIP 로드


In [7]:
@dataclass
class R:
    src: str
    person: str
    label_in_name: str = ''       # 파일명에 적힌 정면/측면
    status: str = 'pass'
    reasons: list = field(default_factory=list)
    has_glasses: Optional[bool] = None
    yaw: float = 0.0
    angle: str = ''               # 실제 측정 각도
    blur: float = 0.0
    fw: int = 0; fh: int = 0
    iw: int = 0; ih: int = 0
    nfaces: int = 0
    emb: Optional[np.ndarray] = None
    final_name: str = ''
    def fail(self, r): self.status='reject'; self.reasons.append(r)


def get_label_from_filename(stem):
    """파일명에서 정면/측면 라벨 추출"""
    if '정면' in stem: return '정면'
    if '측면' in stem: return '측면'
    return '미표기'


def analyze(path):
    p = Path(path)
    parts = p.stem.split('_')
    person = parts[0]
    label_in_name = '미표기'
    for part in parts:
        if '정면' in part: label_in_name = '정면'; break
        if '측면' in part: label_in_name = '측면'; break
    r = R(src=str(path), person=person, label_in_name=label_in_name)
    r.label_in_name = get_label_from_filename(p.stem)

    img = cv2.imread(str(path))
    if img is None: r.fail('로드실패'); return r

    h, w = img.shape[:2]
    r.iw, r.ih = w, h

    # 1. 전체 이미지 해상도
    if w < MIN_IMG or h < MIN_IMG:
        r.fail(f'이미지해상도부족({w}x{h})')

    faces = face_app.get(img)
    r.nfaces = len(faces)

    # 2. 얼굴 감지
    if len(faces) == 0: r.fail('얼굴없음'); return r
    if len(faces) > 1:  r.fail(f'다중얼굴({len(faces)}개)'); return r

    f = faces[0]
    x1,y1,x2,y2 = f.bbox.astype(int)
    fw, fh = x2-x1, y2-y1
    r.fw, r.fh = fw, fh

    # 3. 얼굴 해상도
    if fw < MIN_FACE or fh < MIN_FACE:
        r.fail(f'얼굴해상도부족({fw}x{fh})')

    # 4. 정면/측면 실제 각도 측정 + 파일명 라벨과 비교
    yaw = float(f.pose[1]) if hasattr(f,'pose') else 0.0
    r.yaw = yaw
    r.angle = '정면' if abs(yaw) <= YAW_THR else '측면'
    if r.label_in_name != '미표기' and r.label_in_name != r.angle:
        r.fail(f'각도불일치(라벨={r.label_in_name}/실제={r.angle}/yaw={yaw:.0f}°)')

    # 5. 블러
    crop = img[max(0,y1):min(h,y2), max(0,x1):min(w,x2)]
    if crop.size > 0:
        blur = float(cv2.Laplacian(cv2.cvtColor(crop,cv2.COLOR_BGR2GRAY),cv2.CV_64F).var())
        r.blur = blur
        if blur < BLUR_THR: r.fail(f'블러({blur:.0f})')

    # 6. 안경
    if GLASSES_OK:
        try:
            pil = PILImage.fromarray(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
            r.has_glasses = bool(glasses_clf.process_image(pil))
        except: pass

    # 7. 마스크
    if CLIP_OK:
        try:
            pil = PILImage.fromarray(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
            t = clip_prep(pil).unsqueeze(0)
            with torch.no_grad(): logits,_ = clip_model(t, clip_mask_tok)
            if logits.argmax().item() == 1: r.fail('마스크착용')
        except: pass

    if hasattr(f,'normed_embedding'):
        r.emb = f.normed_embedding.copy()

    return r


def dedup_and_rename(results):
    # pHash 중복
    seen = {}
    for r in results:
        if r.status == 'reject': continue
        try:
            h = str(phash(PILImage.open(r.src)))
            if h in seen: r.fail('pHash중복')
            else: seen[h] = r.src
        except: pass

    # 같은 각도 내 유사 이미지 → MAX_PER_ANGLE 초과 제거
    for angle in ['정면','측면']:
        bucket = [r for r in results if r.status=='pass' and r.angle==angle and r.emb is not None]
        if len(bucket) <= MAX_PER_ANGLE: continue
        embs = np.stack([r.emb for r in bucket])
        sim  = embs @ embs.T
        removed = set()
        for i in range(len(bucket)):
            if i in removed: continue
            cluster = [i]+[j for j in range(i+1,len(bucket)) if j not in removed and sim[i,j]>=SIM_THR]
            if len(cluster) > MAX_PER_ANGLE:
                keep = sorted(cluster, key=lambda x: bucket[x].blur, reverse=True)[:MAX_PER_ANGLE]
                [removed.add(idx) for idx in cluster if idx not in keep]
        for idx in removed: bucket[idx].fail(f'행사중복({angle})')

    # 파일명 부여: [이름]_[정면/측면]_[인덱스].jpg
    cnt = defaultdict(int)
    for r in [r for r in results if r.status=='pass']:
      original_name = Path(r.src).stem
      person_name = original_name.split('_')[0]
      key = (person_name, r.angle)
      cnt[key] += 1
      ext = Path(r.src).suffix.lower() or '.jpg'
      r.final_name = f'{person_name}_{r.angle}_{cnt[key]:03d}{ext}'
    return results


print('✅ 함수 준비 완료')

✅ 함수 준비 완료


In [8]:
clean_root = Path(CLEAN_DIR)
clean_root.mkdir(parents=True, exist_ok=True)
all_results = []

for person, img_paths in sorted(person_images.items()):
    print(f'\n[{person}] {len(img_paths)}장 분석 중...')

    results = [analyze(p) for p in img_paths]
    results = dedup_and_rename(results)

    # 파일 저장
    for r in results:
        if r.status == 'pass':
            dest = clean_root / ('attack' if r.has_glasses else 'enrolled') / r.person
        else:
            key  = r.reasons[0].split('(')[0] if r.reasons else 'unknown'
            dest = clean_root / 'rejected' / key / r.person
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copy2(r.src, dest / (r.final_name or Path(r.src).name))

    passed   = sum(1 for r in results if r.status=='pass' and not r.has_glasses)
    attack   = sum(1 for r in results if r.status=='pass' and r.has_glasses)
    rejected = sum(1 for r in results if r.status=='reject')
    mismatch = sum(1 for r in results if any('각도불일치' in reason for reason in r.reasons))
    print(f'  enrolled={passed}  attack(안경)={attack}  rejected={rejected}  (각도불일치={mismatch})')
    all_results.extend(results)

print('\n🎉 정제 완료!')


[김고은] 150장 분석 중...
  enrolled=27  attack(안경)=0  rejected=123  (각도불일치=25)

[나재민] 150장 분석 중...
  enrolled=75  attack(안경)=0  rejected=75  (각도불일치=30)

[남궁민] 150장 분석 중...
  enrolled=45  attack(안경)=0  rejected=105  (각도불일치=10)

[류진] 150장 분석 중...
  enrolled=47  attack(안경)=0  rejected=103  (각도불일치=37)

[마동석] 150장 분석 중...
  enrolled=10  attack(안경)=0  rejected=140  (각도불일치=25)

[박보검] 150장 분석 중...
  enrolled=45  attack(안경)=0  rejected=105  (각도불일치=11)

[박지훈] 150장 분석 중...
  enrolled=35  attack(안경)=0  rejected=115  (각도불일치=29)

[뷔] 150장 분석 중...
  enrolled=93  attack(안경)=0  rejected=57  (각도불일치=11)

[비투비 이민혁] 150장 분석 중...
  enrolled=46  attack(안경)=0  rejected=104  (각도불일치=33)

[설윤] 150장 분석 중...
  enrolled=58  attack(안경)=0  rejected=92  (각도불일치=20)

[송강호] 150장 분석 중...
  enrolled=37  attack(안경)=0  rejected=113  (각도불일치=17)

[수지] 150장 분석 중...
  enrolled=56  attack(안경)=0  rejected=94  (각도불일치=38)

[유해진] 150장 분석 중...
  enrolled=27  attack(안경)=0  rejected=123  (각도불일치=29)

[이영애] 150장 분석 중...
  enrolled=50  attack(안

## Step 6. 결과 요약

In [9]:
total    = len(all_results)
enrolled = sum(1 for r in all_results if r.status=='pass' and not r.has_glasses)
attack   = sum(1 for r in all_results if r.status=='pass' and r.has_glasses)
rejected = sum(1 for r in all_results if r.status=='reject')

print('='*55)
print(f'  전체 처리   {total}장')
print(f'  enrolled    {enrolled}장   → clean/enrolled/')
print(f'  attack      {attack}장    → clean/attack/  (안경 착용)')
print(f'  rejected    {rejected}장   → clean/rejected/')
print(f'  통과율      {(enrolled+attack)/total*100:.1f}%')
print('='*55)

reason_cnt = defaultdict(int)
for r in all_results:
    for reason in r.reasons:
        reason_cnt[reason.split('(')[0]] += 1

print('\n탈락 사유:')
for k,v in sorted(reason_cnt.items(), key=lambda x:-x[1]):
    bar = '█' * min(v, 40)
    print(f'  {k:<20} {v:>4}장  {bar}')

# 인물별 요약
print('\n인물별 결과:')
print(f'  {"이름":<12} {"통과":>6} {"탈락":>6} {"안경":>6}')
print('  ' + '-'*32)
for person in sorted(person_images.keys()):
    pr = [r for r in all_results if r.person==person]
    p  = sum(1 for r in pr if r.status=='pass' and not r.has_glasses)
    a  = sum(1 for r in pr if r.status=='pass' and r.has_glasses)
    rej= sum(1 for r in pr if r.status=='reject')
    print(f'  {person:<12} {p:>6} {rej:>6} {a:>6}')

# CSV 저장
with open(REPORT,'w',newline='',encoding='utf-8-sig') as f:
    w = csv.writer(f)
    w.writerow(['person','src','final_name','status','reasons',
                'label_in_name','angle_actual','yaw','blur',
                'fw','fh','iw','ih','has_glasses','nfaces'])
    for r in all_results:
        w.writerow([r.person, Path(r.src).name, r.final_name,
                    r.status, '|'.join(r.reasons),
                    r.label_in_name, r.angle, f'{r.yaw:.1f}', f'{r.blur:.0f}',
                    r.fw, r.fh, r.iw, r.ih, r.has_glasses, r.nfaces])
print(f'\n✅ 리포트 저장: {REPORT}')

  전체 처리   3000장
  enrolled    1015장   → clean/enrolled/
  attack      0장    → clean/attack/  (안경 착용)
  rejected    1985장   → clean/rejected/
  통과율      33.8%

탈락 사유:
  행사중복                  669장  ████████████████████████████████████████
  각도불일치                 526장  ████████████████████████████████████████
  얼굴해상도부족               498장  ████████████████████████████████████████
  다중얼굴                  279장  ████████████████████████████████████████
  얼굴없음                  166장  ████████████████████████████████████████
  마스크착용                 140장  ████████████████████████████████████████
  블러                     54장  ████████████████████████████████████████
  pHash중복                 5장  █████

인물별 결과:
  이름               통과     탈락     안경
  --------------------------------
  김고은              27    123      0
  나재민              75     75      0
  남궁민              45    105      0
  류진               47    103      0
  마동석              10    140      0
  박보검              45    105      0
  박지훈

## Step 7. 다운로드

In [10]:
import shutil
from google.colab import files

print('zip 압축 중...')
shutil.make_archive('/content/clean', 'zip', CLEAN_DIR)
print('✅ 압축 완료\n')

files.download(REPORT)                 # report.csv -> 나중에 왜 탈락했는지 각도 불일치 개수 등 알수있어서 파라미터 튜닝할때 참고하기 좋음
files.download('/content/clean.zip')   # 정제 결과

zip 압축 중...
✅ 압축 완료



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>